# SPM 3405: Machine Learning — Lecture 8 notebook
## Unsupervised learning

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will:
1. Summarise 8 pump-station sensors with **principal components analysis (PCA)**.
2. Build a fault monitor from normal data only, using the **PCA reconstruction error**.
3. Group 300 electricity customers by their daily load profile with **$k$-means** and **hierarchical clustering**.

Reading: ISLP 12.1, 12.2 and 12.4. Run each cell with **Shift + Enter**, from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from scipy.cluster.hierarchy import linkage, dendrogram

rng = np.random.default_rng(405)   # fixed seed: everyone gets the same numbers as the slides

## Part 1: A pump station with 8 sensors

In normal operation all 8 sensors are driven by two hidden factors: the **load** on the pump and the **ambient temperature**.

In [ ]:
names = ["flow", "discharge pressure", "suction pressure", "motor current", "motor power",
         "bearing temp", "motor temp", "oil temp"]
L = np.array([[1.0, 0.0], [0.9, 0.1], [-0.7, 0.0], [0.95, 0.15], [0.95, 0.1], [0.4, 0.8], [0.5, 0.8], [0.2, 0.9]])
def pump(n):
    F = rng.normal(0, 1, (n, 2))                   # load, ambient temperature
    return F @ L.T + rng.normal(0, 0.3, (n, 8))    # each sensor, plus its own noise

X = pump(500)                                      # 500 normal readings
scaler = StandardScaler().fit(X); Z = scaler.transform(X)
pca = PCA().fit(Z)
pve = pca.explained_variance_ratio_
print("variance explained by each component (%):", np.round(100 * pve, 1))
print("first two together: %.1f%%" % (100 * pve[:2].sum()))

plt.plot(range(1, 9), 100 * pve, "o-", label="each"); plt.plot(range(1, 9), 100 * np.cumsum(pve), "s-", label="cumulative")
plt.xlabel("component"); plt.ylabel("variance explained (%)"); plt.legend(); plt.show()

In [ ]:
for j, name in enumerate(names):
    print(f"{name:20s} PC1 {pca.components_[0, j]:+.2f}   PC2 {pca.components_[1, j]:+.2f}")
scores = pca.transform(Z)[:, :2]
plt.scatter(scores[:, 0], scores[:, 1], s=8); plt.xlabel("PC1 score"); plt.ylabel("PC2 score"); plt.show()

**Questions**
1. Describe PC1 and PC2 in words, from the loadings. Why does suction pressure have the opposite sign on PC1?
2. The sign of a whole component can flip (all loadings $\times -1$) between computers. Why does this not matter?

### A fault monitor built from normal data only

Test data: 900 normal readings and 100 readings with a **partial blockage** (motor current up, flow down).

In [ ]:
X_normal = pump(900); X_block = pump(100)
X_block[:, 3] += 1.2; X_block[:, 0] -= 0.8          # current up, flow down
X_test = np.vstack([X_normal, X_block]); y_test = np.r_[np.zeros(900), np.ones(100)]
Z_test = scaler.transform(X_test)

pca2 = PCA(2).fit(Z)                                  # fitted on NORMAL data only
def recon_error(Zx):
    return ((Zx - pca2.inverse_transform(pca2.transform(Zx)))**2).sum(axis=1)

err = recon_error(Z_test)
print("AUC, PCA reconstruction error:     ", round(roc_auc_score(y_test, err), 3))
print("AUC, most extreme single sensor:   ", round(roc_auc_score(y_test, np.abs(Z_test).max(axis=1)), 3))
threshold = np.percentile(recon_error(Z), 99)         # 1% false alarms on normal training data
alarm = err > threshold
print(f"alarm threshold {threshold:.2f}: blockages caught {alarm[y_test == 1].mean():.0%}, false alarms {alarm[y_test == 0].mean():.1%}")

plt.hist(err[y_test == 0], bins=40, alpha=0.6, label="normal"); plt.hist(err[y_test == 1], bins=40, alpha=0.6, label="blockage")
plt.axvline(threshold, color="k", ls="--"); plt.xlabel("reconstruction error"); plt.legend(); plt.show()

**Question 3.** Why does no single sensor detect the blockage well, while the reconstruction error does?

## Part 2: Grouping electricity customers

300 customers, 24 hourly readings each. There are three hidden types (households, offices/shops, night businesses) — the algorithm is not told.

In [ ]:
h = np.arange(24)
def profile(kind):
    if kind == 0: return 0.3 + 0.4 * np.exp(-0.5 * ((h - 7) / 1.5)**2) + 1.0 * np.exp(-0.5 * ((h - 20) / 2)**2)   # household
    if kind == 1: return 0.2 + 1.0 / (1 + np.exp(-(h - 7.5) * 2)) - 1.0 / (1 + np.exp(-(h - 17.5) * 2))          # office/shop
    return 0.3 + 0.9 * (1 / (1 + np.exp(-(h - 19) * 2)) + 1 / (1 + np.exp((h - 5) * 2)))                         # night business

n = 300
kind = rng.choice(3, n, p=[0.6, 0.3, 0.1])          # hidden truth, used only to check the result
size = rng.lognormal(0, 0.5, n)                     # big and small customers
P = np.array([size[i] * profile(kind[i]) * (1 + rng.normal(0, 0.15, 24)) for i in range(n)])   # kWh
shape = P / P.sum(axis=1, keepdims=True)            # share of the day's energy in each hour

within = [KMeans(k, n_init=20, random_state=0).fit(shape).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), 100 * np.array(within) / within[0], "o-")
plt.xlabel("number of clusters k"); plt.ylabel("within-cluster variation (% of k=1)"); plt.show()

In [ ]:
km = KMeans(3, n_init=20, random_state=0).fit(shape)
print("cluster sizes:", np.bincount(km.labels_))
print("agreement with the true types (adjusted Rand index):", round(adjusted_rand_score(kind, km.labels_), 2))
for c in range(3):
    plt.plot(h, 100 * km.cluster_centers_[c], label=f"cluster {c} ({np.sum(km.labels_ == c)} customers)")
plt.xlabel("hour"); plt.ylabel("% of daily energy"); plt.legend(); plt.show()

raw = KMeans(3, n_init=20, random_state=0).fit(P)
print("on raw kWh instead: sizes", np.bincount(raw.labels_), " agreement", round(adjusted_rand_score(kind, raw.labels_), 2))

In [ ]:
for link in ["complete", "average", "single"]:
    lab = AgglomerativeClustering(3, linkage=link).fit_predict(shape)
    print(f"hierarchical, {link:8s} linkage: sizes {np.bincount(lab)}  agreement {adjusted_rand_score(kind, lab):.2f}")

idx = np.r_[np.where(kind == 0)[0][:20], np.where(kind == 1)[0][:12], np.where(kind == 2)[0][:8]]
plt.figure(figsize=(10, 3.5))
dendrogram(linkage(shape[idx], "complete"), labels=[["H", "O", "N"][k] for k in kind[idx]])
plt.title("40 customers (H household, O office/shop, N night business)"); plt.show()

**Questions**
4. Where is the elbow in the within-cluster plot? Does it agree with the three true types?
5. Clustering the raw kWh gives very different groups. What do those groups describe? When might they be the ones you want?
6. Run $k$-means with $k = 4$. What does the extra cluster look like?
7. **Your group project:** prepare the 10-minute presentation described on the slide "Group project presentations".

## Your turn: group tasks

Work on these in your group. Write the code in the empty cells and answer each question in a text cell below it,
in your own words. Submit **one notebook per group** on Google Classroom, with every cell run.
Every member must be able to explain every line.

The tasks reuse `Z`, `pca`, `pump`, `scaler`, `names`, `shape`, `kind` and the imports from above, so run the whole notebook first.

### Task 1: PCA by hand

scikit-learn's `PCA` hides a short calculation. Do it yourself for the 500 standardised pump readings `Z`.

(a) Compute the $8 \times 8$ covariance matrix $\mathbf C = \mathbf Z^{\mathsf T}\mathbf Z/(n-1)$.

(b) Find its eigenvalues and eigenvectors with `np.linalg.eigh`, and sort them from the largest eigenvalue down.

(c) Show that the eigenvalues divided by their sum equal `pca.explained_variance_ratio_`, and that the first two eigenvectors equal
`pca.components_[:2]` up to a sign. What does each eigenvalue measure?

In [ ]:
# Task 1: your code here

### Task 2: How many components should the monitor keep?

The monitor in Part 1 keeps 2 components. Test that choice against **two** kinds of fault:
the partial blockage of Part 1, and a **drifting sensor**: the bearing-temperature sensor reads 1.5 standard deviations too high,
while the pump itself is fine.

(a) Make 900 normal test readings, 100 blockages (as in Part 1) and 100 drifting-sensor readings (add `1.5 * scaler.scale_[5]` to column 5).

(b) For $k = 1, 2, \dots, 6$ components: fit `PCA(k)` on the normal training data `Z`, set the alarm threshold at the 99th percentile of
the training reconstruction error, and record the fraction of blockages caught, of drifts caught, and of normal readings that raise an alarm.
Show the results as a table.

(c) Which $k$ would you choose? Why does keeping too many components make the monitor blind?

In [ ]:
# Task 2: your code here

### Task 3: $k$-means by hand

(a) Write your own $k$-means (Lloyd's algorithm) for the customer shapes `shape`:
start from $k$ customers chosen at random as centres; then repeat (1) assign each customer to the nearest centre, (2) move each centre to
the mean of its customers, until no customer changes cluster. Return the labels and the within-cluster sum of squares.

(b) Run it with $k = 3$ from 10 different random starts. Print the within-cluster sum of squares and the agreement with the true types
(`adjusted_rand_score(kind, labels)`) for each start. Then run it with $k = 5$ from 10 starts and print the within-cluster sum of squares.

(c) With $k = 3$, do all starts give the same answer? With $k = 5$? Explain the difference.
Why does scikit-learn's `KMeans` use `n_init=20`, and which of the 20 answers does it keep?

In [ ]:
# Task 3: your code here